# Python Loops — A Deep Dive

> **Series:** Learn the Basics | **Previous:** Working with Strings

Loops are the engine of repetition in Python. This notebook goes deep:
from `range()` internals to `itertools`, lazy generators, and custom iterators.

---

## Table of Contents
1. [for Loops](#1.-for-Loops)
2. [range() in Depth](#2.-range()-in-Depth)
3. [Looping with enumerate()](#3.-Looping-with-enumerate())
4. [Looping with zip()](#4.-Looping-with-zip())
5. [Iterating over Data Structures](#5.-Iterating-over-Data-Structures)
6. [Nested Loops](#6.-Nested-Loops)
7. [while Loops](#7.-while-Loops)
8. [break, continue, pass](#8.-break,-continue,-pass)
9. [Loop else Clause](#9.-Loop-else-Clause)
10. [Comprehensions — Deep Dive](#10.-Comprehensions-—-Deep-Dive)
11. [itertools Module](#11.-itertools-Module)
12. [Iterators & Generators](#12.-Iterators-&-Generators)
13. [Performance Tips](#13.-Performance-Tips)
14. [Quick Reference Card](#14.-Quick-Reference-Card)


---
## 1. for Loops

Python's `for` loop iterates over any **iterable** object — anything that
implements the iterator protocol (`__iter__` and `__next__`).

```python
for variable in iterable:
    # body executes once per element
```

Built-in iterables: `list`, `tuple`, `str`, `bytes`, `range`, `dict`,
`set`, `frozenset`, `file`, generator objects, and more.


In [ ]:
# Iterating over common types
print("--- list ---")
for x in [10, 20, 30]:
    print(x, end=" ")
print()

print("--- string ---")
for ch in "Python":
    print(ch, end=" ")
print()

print("--- tuple ---")
for item in ("a", "b", "c"):
    print(item, end=" ")
print()

print("--- set (unordered) ---")
for item in {3, 1, 4, 1, 5}:   # no guaranteed order, no duplicates
    print(item, end=" ")
print()


In [ ]:
# Unpacking inside for loops (tuple unpacking)
points = [(1, 2), (3, 4), (5, 6)]
for x, y in points:
    print(f"  ({x}, {y}) -> distance = {(x**2 + y**2)**0.5:.2f}")

print()

# Three-value unpacking
records = [("Alice", 30, "Engineer"), ("Bob", 25, "Designer")]
for name, age, role in records:
    print(f"  {name:<6} | {age} | {role}")


In [ ]:
# Starred unpacking in for loops
data = [
    ("Alice", 90, 85, 92, 88),
    ("Bob",   70, 75, 80, 65),
]

for name, first, *rest in data:
    avg = sum(rest) / len(rest) if rest else 0
    print(f"  {name}: first={first}, other scores={rest}, avg={avg:.1f}")


In [ ]:
# The _ convention for unused variables
# Run something N times without caring about the index
results = []
import random; random.seed(42)
for _ in range(5):
    results.append(random.randint(1, 100))
print(f"5 random numbers: {results}")

# Unpack but ignore some fields
rows = [(1, "Alice", "unused"), (2, "Bob", "unused")]
for id_, name, _ in rows:
    print(f"  id={id_}, name={name}")


---
## 2. range() in Depth

`range(start, stop, step)` produces an **immutable sequence** of integers.
It is **lazy** — it does not store all values in memory.

| Call | Sequence produced |
|------|-------------------|
| `range(n)` | `0, 1, 2, ..., n-1` |
| `range(a, b)` | `a, a+1, ..., b-1` |
| `range(a, b, s)` | `a, a+s, a+2s, ...` (up to but not including `b`) |
| `range(b, a, -s)` | Counts **down** from `b-1` to `a` |


In [ ]:
# range() variants
print("range(5)          :", list(range(5)))
print("range(2, 8)       :", list(range(2, 8)))
print("range(0, 20, 3)   :", list(range(0, 20, 3)))
print("range(10, 0, -2)  :", list(range(10, 0, -2)))
print("range(5, -1, -1)  :", list(range(5, -1, -1)))


In [ ]:
# range is lazy and memory-efficient
import sys
r = range(1_000_000)
print(f"range(1_000_000) memory: {sys.getsizeof(r)} bytes")
print(f"list(range(1000)) memory: {sys.getsizeof(list(range(1000)))} bytes")
print(f"len(r)  = {len(r)}")
print(f"r[0]    = {r[0]}")
print(f"r[-1]   = {r[-1]}")
print(f"r[::2]  = {list(r[::2])[:6]}...  (sliceable!)")
print(f"500 in r: {500 in r}")


In [ ]:
# Common patterns with range()

# Sum 1..100
total = sum(range(1, 101))
print(f"sum(1..100)   = {total}")

# Index-based access
fruits = ["apple", "banana", "cherry"]
for i in range(len(fruits)):
    print(f"  [{i}] {fruits[i]}")

print()
# But enumerate() is more Pythonic (see next section)
for i, fruit in enumerate(fruits):
    print(f"  [{i}] {fruit}")


In [ ]:
# Reverse iteration
items = ["a", "b", "c", "d", "e"]

# Option 1: range with step -1
print("range(len-1, -1, -1):")
for i in range(len(items) - 1, -1, -1):
    print(f"  {i}: {items[i]}", end="  ")
print()

# Option 2: reversed() — more readable
print("reversed():")
for item in reversed(items):
    print(item, end="  ")
print()

# Option 3: slicing [::-1] — creates a copy
print("slice [::-1]:")
for item in items[::-1]:
    print(item, end="  ")
print()


---
## 3. Looping with enumerate()

`enumerate(iterable, start=0)` wraps any iterable and yields
`(index, value)` tuples. Always prefer this over `range(len(...))`.

```python
for i, value in enumerate(iterable, start=0):
    ...
```


In [ ]:
fruits = ["apple", "banana", "cherry", "date"]

# Default: starts at 0
for i, fruit in enumerate(fruits):
    print(f"  {i}: {fruit}")

print()
# Start at 1
for i, fruit in enumerate(fruits, start=1):
    print(f"  {i}. {fruit}")


In [ ]:
# enumerate with strings
for i, ch in enumerate("PYTHON"):
    print(f"  s[{i}] = {ch!r}")


In [ ]:
# Practical: find all indices of a value
scores = [90, 75, 90, 88, 75, 90]
target = 90
indices = [i for i, s in enumerate(scores) if s == target]
print(f"Score {target} found at indices: {indices}")

# Practical: update elements by index
data = ["alice", "bob", "carol"]
for i, name in enumerate(data):
    data[i] = name.capitalize()
print(f"Capitalized: {data}")


In [ ]:
# Building a manual enumerate() to understand internals
def my_enumerate(iterable, start=0):
    index = start
    for item in iterable:
        yield index, item
        index += 1

for i, v in my_enumerate(["x", "y", "z"], start=10):
    print(f"  {i}: {v}")


---
## 4. Looping with zip()

`zip(*iterables)` combines multiple iterables element-by-element,
stopping at the **shortest**. Use `itertools.zip_longest` to fill gaps.

```python
for a, b, c in zip(list1, list2, list3):
    ...
```


In [ ]:
names  = ["Alice", "Bob", "Carol"]
scores = [92, 85, 78]
grades = ["A", "B", "C"]

for name, score, grade in zip(names, scores, grades):
    print(f"  {name:<6} | {score} | {grade}")

print()
# zip() returns an iterator of tuples
zipped = list(zip(names, scores))
print(f"list(zip(...)): {zipped}")


In [ ]:
# zip stops at the shortest iterable
a = [1, 2, 3, 4, 5]
b = ["x", "y", "z"]

print("zip (stops at shortest):")
for pair in zip(a, b):
    print(f"  {pair}")

print()
# zip_longest pads with a fill value
from itertools import zip_longest
print("zip_longest (fills missing):")
for pair in zip_longest(a, b, fillvalue="?"):
    print(f"  {pair}")


In [ ]:
# Unzip with zip(*iterable)
pairs = [(1, "a"), (2, "b"), (3, "c"), (4, "d")]

numbers, letters = zip(*pairs)
print(f"numbers : {numbers}")
print(f"letters : {letters}")

# Transposing a matrix with zip
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
transposed = list(zip(*matrix))
print(f"original  : {matrix}")
print(f"transposed: {[list(row) for row in transposed]}")


In [ ]:
# zip() with enumerate()
students = ["Alice", "Bob", "Carol"]
marks    = [88, 72, 95]

for i, (student, mark) in enumerate(zip(students, marks), start=1):
    status = "PASS" if mark >= 75 else "FAIL"
    print(f"  {i}. {student:<6} {mark:>3}  [{status}]")


---
## 5. Iterating over Data Structures

Every Python collection supports the `for` loop.


In [ ]:
# --- Dictionaries ---
person = {"name": "Alice", "age": 30, "city": "Paris"}

print("Keys (default):")
for k in person:
    print(f"  {k}")

print("Values:")
for v in person.values():
    print(f"  {v}")

print("Items (key-value pairs):")
for k, v in person.items():
    print(f"  {k}: {v}")


In [ ]:
# Modifying a dict while iterating (copy keys first!)
scores = {"Alice": 90, "Bob": 40, "Carol": 85, "Dave": 35}

# Safe: iterate over a copy of keys
for name in list(scores.keys()):
    if scores[name] < 50:
        del scores[name]

print(f"After removing failures: {scores}")


In [ ]:
# --- Sets ---
colors = {"red", "green", "blue", "red", "yellow"}
print(f"Set (unordered, unique): {colors}")

for color in sorted(colors):   # sorted() for reproducible output
    print(f"  {color}")


In [ ]:
# --- Nested data structures ---
students = [
    {"name": "Alice", "grades": [90, 85, 92]},
    {"name": "Bob",   "grades": [70, 75, 68]},
    {"name": "Carol", "grades": [88, 91, 95]},
]

for student in students:
    avg = sum(student["grades"]) / len(student["grades"])
    best = max(student["grades"])
    print(f"  {student['name']:<6} avg={avg:.1f}  best={best}")


In [ ]:
# --- Files (line-by-line, memory-efficient) ---
import io

# Simulate a file with StringIO
fake_file = io.StringIO("line one\nline two\nline three\n")

for i, line in enumerate(fake_file, start=1):
    print(f"  {i}: {line.rstrip()}")

# In real code:
# with open('data.txt') as f:
#     for line in f:
#         process(line)


---
## 6. Nested Loops

A loop inside another loop. Inner loop completes **all** iterations
for each single iteration of the outer loop.

> Total iterations = outer_count × inner_count — watch out for O(n²) complexity.


In [ ]:
# Multiplication table
size = 5
header = "   " + "  ".join(f"{j:2d}" for j in range(1, size + 1))
print(header)
print("-" * len(header))
for i in range(1, size + 1):
    row = f"{i:2d} | " + "  ".join(f"{i*j:2d}" for j in range(1, size + 1))
    print(row)


In [ ]:
# Pattern printing with nested loops
n = 5

print("Right triangle:")
for i in range(1, n + 1):
    print("  " + "* " * i)

print("Pyramid:")
for i in range(1, n + 1):
    print(" " * (n - i) + "* " * i)

print("Checkerboard (5x5):")
for i in range(5):
    row = "".join("# " if (i + j) % 2 == 0 else ". " for j in range(5))
    print(" " + row)


In [ ]:
# Flattening nested structures
nested = [[1, 2, 3], [4, 5], [6, 7, 8, 9]]

# Traditional nested loop
flat_loop = []
for sublist in nested:
    for item in sublist:
        flat_loop.append(item)

# List comprehension (equivalent)
flat_comp = [item for sublist in nested for item in sublist]

# itertools.chain (most efficient)
from itertools import chain
flat_chain = list(chain.from_iterable(nested))

print(f"loop  : {flat_loop}")
print(f"comp  : {flat_comp}")
print(f"chain : {flat_chain}")


In [ ]:
# Breaking out of nested loops cleanly

# Option 1: flag variable
found = False
for i in range(5):
    for j in range(5):
        if i * j == 12:
            print(f"Option 1: {i} x {j} = 12")
            found = True
            break
    if found:
        break

# Option 2: use a function (cleanest)
def find_product(target, n):
    for i in range(n):
        for j in range(n):
            if i * j == target:
                return i, j
    return None

result = find_product(12, 10)
print(f"Option 2: {result[0]} x {result[1]} = 12")

# Option 3: exception (uncommon but valid)
class Found(Exception): pass

try:
    for i in range(5):
        for j in range(5):
            if i * j == 12:
                raise Found(i, j)
except Found as e:
    print(f"Option 3: {e.args[0]} x {e.args[1]} = 12")


---
## 7. while Loops

Repeats a block as long as a condition is truthy.
Use `while` when the **number of iterations is unknown** in advance.

```python
while condition:
    # body
```


In [ ]:
# Basic while
n = 1
while n < 100:
    n *= 2
print(f"First power of 2 >= 100: {n}")


In [ ]:
# do-while pattern (Python has no do-while keyword)
# Emulate with while True + break
import random; random.seed(7)

roll = 0
attempts = 0
while True:                        # always enter at least once
    roll = random.randint(1, 6)
    attempts += 1
    print(f"  Roll {attempts}: {roll}")
    if roll == 6:
        break
print(f"Got 6 after {attempts} attempt(s)")


In [ ]:
# Walrus operator in while (Python 3.8+)
import re

# Classic
tokens_classic = []
text = "abc 123 def 456 ghi"
pos = 0
while pos < len(text):
    m = re.match(r"\d+", text[pos:])
    if m:
        tokens_classic.append(m.group())
        pos += m.end()
    else:
        pos += 1
print(f"classic: {tokens_classic}")

# Walrus
data = iter([3, 7, 0, 2, 5, 0, 9])
batch = []
while (val := next(data, None)) is not None:
    if val == 0:
        break
    batch.append(val)
print(f"walrus batch before 0: {batch}")


In [ ]:
# Classic algorithms with while

# GCD (Euclidean algorithm)
def gcd(a, b):
    while b:
        a, b = b, a % b
    return a

print(f"gcd(48, 18) = {gcd(48, 18)}")
print(f"gcd(100, 75) = {gcd(100, 75)}")

# Newton-Raphson square root
def sqrt_newton(n, tol=1e-10):
    x = float(n)
    while True:
        root = (x + n / x) / 2
        if abs(root - x) < tol:
            return root
        x = root

for v in [2, 9, 144, 1234567]:
    print(f"  sqrt({v}) = {sqrt_newton(v):.8f}  (math: {v**0.5:.8f})")


---
## 8. break, continue, pass

| Statement | What it does |
|-----------|-------------|
| `break` | Immediately exit the nearest enclosing loop |
| `continue` | Skip the rest of the current iteration; go to next |
| `pass` | Do nothing — syntactic placeholder |


In [ ]:
# --- break ---
print("First prime > 50:")
def is_prime(n):
    if n < 2: return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0: return False
    return True

for n in range(51, 200):
    if is_prime(n):
        print(f"  {n}")
        break

print()
# break in while
limit, total = 100, 0
n = 1
while True:
    total += n
    if total >= limit:
        print(f"Sum reached {total} after adding {n}")
        break
    n += 1


In [ ]:
# --- continue ---
print("Odd numbers 1-10:")
for i in range(1, 11):
    if i % 2 == 0:
        continue
    print(i, end=" ")
print()

# Skip invalid data
raw = ["42", "hello", "7", None, "3.14", "", "99"]
valid = []
for item in raw:
    if not item:         # None or empty string
        continue
    try:
        valid.append(float(item))
    except ValueError:
        continue         # skip non-numeric strings
print(f"Valid numbers: {valid}")


In [ ]:
# --- pass ---
# Placeholder in loop body (useful when sketching structure)
for _ in range(5):
    pass   # TODO: implement later

# Silent exception handler
for s in ["1", "two", "3"]:
    try:
        print(int(s), end=" ")
    except ValueError:
        pass   # intentionally ignore non-numeric
print()


---
## 9. Loop else Clause

The `else` block of a loop runs **only if the loop finished normally**
(i.e. was NOT terminated by `break`).

```python
for x in iterable:
    if condition: break
else:
    # runs ONLY when break was never hit
    ...
```

This is the cleanest way to implement **search loops** in Python.


In [ ]:
# Prime checker using for...else
def is_prime_else(n):
    if n < 2: return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            break           # composite — stop searching
    else:
        return True         # no divisor found
    return False

primes = [n for n in range(2, 50) if is_prime_else(n)]
print(f"Primes < 50: {primes}")


In [ ]:
# Search with meaningful 'not found' action
inventory = [
    {"sku": "A001", "qty": 5},
    {"sku": "B002", "qty": 0},
    {"sku": "C003", "qty": 12},
]

def find_item(sku):
    for item in inventory:
        if item["sku"] == sku:
            if item["qty"] > 0:
                print(f"  Found {sku}: {item['qty']} in stock")
            else:
                print(f"  Found {sku}: OUT OF STOCK")
            break
    else:
        print(f"  {sku}: not found in inventory")

find_item("B002")
find_item("C003")
find_item("X999")


In [ ]:
# while...else — retry with fallback
import random; random.seed(3)

MAX_RETRIES = 4
attempt = 0

while attempt < MAX_RETRIES:
    attempt += 1
    success = random.random() > 0.7   # 30% success rate
    print(f"  Attempt {attempt}: {'OK' if success else 'FAILED'}")
    if success:
        break
else:
    print(f"  All {MAX_RETRIES} retries exhausted!")


---
## 10. Comprehensions — Deep Dive

Comprehensions are concise, Pythonic loop expressions that build collections.

| Form | Syntax | Result |
|------|--------|--------|
| List | `[expr for x in it if cond]` | `list` |
| Dict | `{k: v for x in it if cond}` | `dict` |
| Set | `{expr for x in it if cond}` | `set` |
| Generator | `(expr for x in it if cond)` | lazy `generator` |


In [ ]:
# --- List comprehension ---
squares  = [x**2 for x in range(1, 11)]
evens    = [x for x in range(20) if x % 2 == 0]
matrix   = [[i * j for j in range(1, 5)] for i in range(1, 5)]

print(f"squares : {squares}")
print(f"evens   : {evens}")
print("matrix  :")
for row in matrix:
    print(f"  {row}")


In [ ]:
# Filtering + transforming in one step
words = ["apple", "Banana", "cherry", "Date", "elderberry"]

# Words longer than 5 chars, uppercased
result = [w.upper() for w in words if len(w) > 5]
print(f"long words (upper): {result}")

# Conditional expression inside comprehension
labels = [f"{x} (even)" if x % 2 == 0 else f"{x} (odd)" for x in range(6)]
print(f"labels: {labels}")


In [ ]:
# --- Dict comprehension ---
# Square mapping
sq_map = {x: x**2 for x in range(1, 6)}
print(f"sq_map   : {sq_map}")

# Invert a dictionary
original = {"a": 1, "b": 2, "c": 3}
inverted = {v: k for k, v in original.items()}
print(f"inverted : {inverted}")

# Filter and transform dict
prices = {"apple": 1.2, "banana": 0.5, "truffle": 89.0, "avocado": 2.5}
affordable = {k: v for k, v in prices.items() if v < 5}
print(f"affordable: {affordable}")


In [ ]:
# --- Set comprehension ---
text = "the quick brown fox jumps over the lazy dog"
unique_len = {len(w) for w in text.split()}
print(f"unique word lengths: {sorted(unique_len)}")

# Unique vowels in a string
vowels = {ch.lower() for ch in text if ch.lower() in "aeiou"}
print(f"unique vowels      : {sorted(vowels)}")


In [ ]:
# --- Generator expressions ---
import sys

# Memory comparison
N = 100_000
list_mem = sys.getsizeof([x**2 for x in range(N)])
gen_mem  = sys.getsizeof((x**2 for x in range(N)))
print(f"list comprehension : {list_mem:,} bytes")
print(f"generator expression: {gen_mem:,} bytes")

# Generator is consumed once — use when you only need one pass
gen = (x**2 for x in range(10))
print(f"\nnext(): {next(gen)}, {next(gen)}, {next(gen)}")
print(f"sum of rest: {sum(gen)}")   # consumes remaining
print(f"now empty: {list(gen)}")    # [] — exhausted


In [ ]:
# Nested comprehension — flatten & filter
data = [[1, -2, 3], [-4, 5, -6], [7, 8, -9]]

positives = [x for row in data for x in row if x > 0]
print(f"positives: {positives}")

# Equivalent loop version
positives_loop = []
for row in data:
    for x in row:
        if x > 0:
            positives_loop.append(x)
print(f"loop ver : {positives_loop}")


---
## 11. itertools Module

`itertools` is a standard-library module of **fast, memory-efficient**
iterator building blocks. All functions return lazy iterators.

| Function | Description |
|----------|-------------|
| `count(n, step)` | Infinite counter |
| `cycle(it)` | Cycle infinitely |
| `repeat(val, n)` | Repeat a value |
| `chain(*its)` | Concatenate iterables |
| `chain.from_iterable(it)` | Flatten one level |
| `islice(it, n)` | Lazy slicing |
| `takewhile(pred, it)` | Take while predicate holds |
| `dropwhile(pred, it)` | Drop while predicate holds |
| `filterfalse(pred, it)` | Filter where pred is False |
| `compress(it, sel)` | Select elements by boolean mask |
| `starmap(fn, args)` | Map with unpacked args |
| `accumulate(it, fn)` | Running totals |
| `groupby(it, key)` | Group consecutive elements |
| `product(*its)` | Cartesian product |
| `permutations(it, r)` | All ordered r-length arrangements |
| `combinations(it, r)` | All unordered r-length subsets |
| `combinations_with_replacement` | Subsets with repetition |
| `pairwise(it)` | Consecutive overlapping pairs (3.10+) |


In [ ]:
import itertools as it

# --- Infinite iterators ---
print("count(10, 3) first 5:",
      list(it.islice(it.count(10, 3), 5)))

print("cycle([A,B,C]) first 8:",
      list(it.islice(it.cycle(["A", "B", "C"]), 8)))

print("repeat('*', 5):",
      list(it.repeat("*", 5)))


In [ ]:
# --- chain ---
print("chain:", list(it.chain([1, 2], [3, 4], [5])))

nested = [[1, 2], [3, 4], [5, 6, 7]]
print("chain.from_iterable:", list(it.chain.from_iterable(nested)))

# --- islice ---
print("islice(range(100), 2, 10, 2):",
      list(it.islice(range(100), 2, 10, 2)))


In [ ]:
# --- takewhile / dropwhile ---
nums = [2, 4, 6, 7, 8, 10, 3]
print("takewhile(even):", list(it.takewhile(lambda x: x % 2 == 0, nums)))
print("dropwhile(even):", list(it.dropwhile(lambda x: x % 2 == 0, nums)))

# --- accumulate ---
import operator
data = [1, 2, 3, 4, 5]
print("accumulate (sum):    ", list(it.accumulate(data)))
print("accumulate (product):", list(it.accumulate(data, operator.mul)))
print("accumulate (max):    ", list(it.accumulate(data, max)))


In [ ]:
# --- groupby ---
# NOTE: groupby works on CONSECUTIVE equal keys — sort first!
words = ["apple", "avocado", "banana", "blueberry", "cherry", "cranberry"]
words.sort()

for letter, group in it.groupby(words, key=lambda w: w[0]):
    print(f"  {letter}: {list(group)}")


In [ ]:
# --- product / permutations / combinations ---

# Cartesian product
suits  = ["♠", "♥", "♦", "♣"]
values = ["A", "K", "Q"]
cards  = list(it.product(values, suits))
print(f"product (first 6): {cards[:6]}  ... total: {len(cards)}")

# Permutations
perms = list(it.permutations("ABC", 2))
print(f"permutations('ABC', 2): {perms}")

# Combinations
combs = list(it.combinations("ABCD", 2))
print(f"combinations('ABCD', 2): {combs}")

# With replacement
combs_r = list(it.combinations_with_replacement("AB", 3))
print(f"comb_with_repl('AB', 3): {combs_r}")


---
## 12. Iterators & Generators

Understanding the **iterator protocol** lets you build custom lazy sequences.

| Concept | Description |
|---------|-------------|
| **Iterable** | Object with `__iter__()` — can be looped over |
| **Iterator** | Object with `__iter__()` + `__next__()` — knows current position |
| **Generator function** | Function with `yield` — creates an iterator automatically |
| **Generator expression** | `(expr for x in it)` — one-liner generator |


In [ ]:
# The iterator protocol
data = [10, 20, 30]

# What 'for x in data' does under the hood:
it_obj = iter(data)         # calls data.__iter__()
print(next(it_obj))         # calls it_obj.__next__() -> 10
print(next(it_obj))         # -> 20
print(next(it_obj))         # -> 30
try:
    print(next(it_obj))     # raises StopIteration
except StopIteration:
    print("StopIteration raised — loop ends")


In [ ]:
# Custom iterator class
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):        # makes it iterable
        return self

    def __next__(self):        # produces next value
        if self.current < 0:
            raise StopIteration
        val = self.current
        self.current -= 1
        return val

for n in Countdown(5):
    print(n, end=" ")
print("GO!")


In [ ]:
# Generator functions — simpler than iterator classes
def countdown_gen(start):
    while start >= 0:
        yield start             # suspends & resumes here
        start -= 1

for n in countdown_gen(5):
    print(n, end=" ")
print("GO!")

# Generators are lazy — values produced on demand
gen = countdown_gen(3)
print(f"type: {type(gen)}")    # <class 'generator'>
print(next(gen))               # 3
print(next(gen))               # 2


In [ ]:
# Infinite generator
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

from itertools import islice
fib_20 = list(islice(fibonacci(), 20))
print(f"First 20 Fibonacci: {fib_20}")

# First Fibonacci > 1000
fib = fibonacci()
while (n := next(fib)) <= 1000:
    pass
print(f"First Fibonacci > 1000: {n}")


In [ ]:
# Generator pipelines — process data lazily
import os

def read_lines(text):             # source
    for line in text.strip().splitlines():
        yield line

def strip_lines(lines):           # transform 1
    for line in lines:
        yield line.strip()

def non_empty(lines):             # filter
    for line in lines:
        if line:
            yield line

def to_upper(lines):              # transform 2
    for line in lines:
        yield line.upper()

raw = """
  hello world  

  python is great  
  learn generators  
"""

pipeline = to_upper(non_empty(strip_lines(read_lines(raw))))
for line in pipeline:
    print(f"  >> {line}")


---
## 13. Performance Tips

| Tip | Why |
|-----|-----|
| Use `for x in iterable` not `for i in range(len(...))` | Avoids index lookup |
| Use `enumerate()` when you need index + value | Cleaner than `range(len(...))` |
| Prefer `join()` over `+=` in string loops | O(n) vs O(n²) |
| Use comprehensions for simple transforms | Faster than `append()` loops |
| Use generators for large data | O(1) memory vs O(n) |
| Use `any()`/`all()` with generators | Short-circuits early |
| Hoist invariants out of inner loops | Avoids repeated computation |
| Use `set`/`dict` for membership tests | O(1) vs O(n) for list |


In [ ]:
import timeit

# 1. List vs set membership
haystack_list = list(range(10_000))
haystack_set  = set(range(10_000))
needle = 9_999

t_list = timeit.timeit(lambda: needle in haystack_list, number=10_000)
t_set  = timeit.timeit(lambda: needle in haystack_set,  number=10_000)
print(f"list 'in': {t_list:.4f}s")
print(f"set  'in': {t_set:.4f}s")
print(f"speedup  : {t_list / t_set:.0f}x")


In [ ]:
# 2. String building: += vs join
N = 5_000

t_plus  = timeit.timeit(
    lambda: [s := "", [s := s + str(i) for i in range(N)], s][-1],
    number=50)

t_join  = timeit.timeit(
    lambda: "".join(str(i) for i in range(N)),
    number=50)

print(f"+= loop : {t_plus:.4f}s")
print(f"join    : {t_join:.4f}s")


In [ ]:
# 3. Hoist invariants out of inner loops
data = list(range(1000))

# Bad: len(data) re-evaluated each iteration
result_bad = []
for i in range(len(data)):
    if data[i] > len(data) // 2:   # len(data) computed N times!
        result_bad.append(data[i])

# Good: compute once
half = len(data) // 2
result_good = [x for x in data if x > half]

print(f"bad  len: {len(result_bad)}")
print(f"good len: {len(result_good)}")
print(f"match   : {result_bad == result_good}")


In [ ]:
# 4. short-circuit with any() / all()
numbers = list(range(1, 10_001))

# any() stops as soon as it finds True
has_even = any(n % 2 == 0 for n in numbers)
print(f"has even  : {has_even}")

# all() stops as soon as it finds False
all_pos  = all(n > 0 for n in numbers)
print(f"all positive: {all_pos}")

# Without short-circuit: would check ALL elements
import timeit
t1 = timeit.timeit(lambda: any(n > 9999 for n in numbers), number=1000)
t2 = timeit.timeit(lambda: any(n > 0    for n in numbers), number=1000)
print(f"any(n>9999): {t1:.4f}s  (scans almost all)")
print(f"any(n>0)   : {t2:.6f}s  (stops at first element)")


---
## 14. Quick Reference Card

All loop patterns in one runnable cell:


In [ ]:
# ==================================================================
# PYTHON LOOPS – QUICK REFERENCE
# ==================================================================

# --- for loop ---
total = 0
for n in range(1, 6):
    total += n
print(f"for sum 1..5 : {total}")

# --- enumerate ---
for i, ch in enumerate("ABC"):
    print(f"  [{i}] {ch}", end="  ")
print()

# --- zip ---
for a, b in zip([1, 2, 3], ["x", "y", "z"]):
    print(f"  ({a},{b})", end="  ")
print()

# --- while ---
n = 1
while n < 50:
    n *= 2
print(f"while first power of 2 >= 50: {n}")

# --- break / continue ---
evens = []
for i in range(20):
    if i > 10: break
    if i % 2 != 0: continue
    evens.append(i)
print(f"break+continue evens<=10: {evens}")

# --- loop else ---
for n in [4, 6, 8, 9]:
    if n % 2 != 0:
        print(f"for-else: found odd {n}")
        break
else:
    print("for-else: all even")

# --- comprehensions ---
sq_list = [x**2 for x in range(1, 6)]
sq_dict = {x: x**2 for x in range(1, 6)}
sq_set  = {x**2 for x in range(1, 6)}
sq_sum  = sum(x**2 for x in range(1, 6))
print(f"list comp : {sq_list}")
print(f"dict comp : {sq_dict}")
print(f"set  comp : {sq_set}")
print(f"gen  sum  : {sq_sum}")

# --- itertools ---
from itertools import chain, accumulate
print(f"chain     : {list(chain([1,2],[3,4],[5]))}")
print(f"accumulate: {list(accumulate([1,2,3,4,5]))}")

# --- generator function ---
def squares_gen(n):
    for i in range(1, n + 1):
        yield i ** 2
print(f"generator : {list(squares_gen(5))}")


---
## Summary

| Tool | Best for |
|------|----------|
| `for x in it` | Iterating any iterable |
| `range(a, b, s)` | Index-based numeric loops |
| `enumerate(it)` | Index + value together |
| `zip(*its)` | Parallel iteration |
| `zip_longest` | Parallel with padding |
| `reversed(it)` | Reverse traversal |
| `while cond` | Unknown iteration count |
| `while True + break` | do-while pattern |
| `break` | Early exit |
| `continue` | Skip current iteration |
| `for...else` | Search loops with not-found action |
| `[x for x in it]` | Build lists concisely |
| `{k:v for ...}` | Build dicts concisely |
| `(x for x in it)` | Lazy iteration, O(1) memory |
| `itertools.*` | Efficient combinatorial & functional iteration |
| `yield` generator | Custom lazy sequences |
| `any()` / `all()` | Short-circuiting existence checks |

---
*Next up: **Functions***
